In [1]:
import cv2,time
from picamera2 import Picamera2
import spidev as SPI
import xgoscreen.LCD_2inch as LCD_2inch
import RPi.GPIO as GPIO
from PIL import Image,ImageDraw,ImageFont
import threading

camera_still=False
class Recode_Video():
    def __init__(self):
        self.display = LCD_2inch.LCD_2inch()
        self.display.Init()
        self.display.clear()
        self.splash = Image.new("RGB",(320,240),"black")
        self.display.ShowImage(self.splash)
        self.draw = ImageDraw.Draw(self.splash)
        self.font = ImageFont.truetype("/home/pi/model/msyh.ttc",15)
        self.picam2=None
        self.camera_still=False
        self.open_camera()

    def open_camera(self):
        if self.picam2==None:
            self.picam2 = Picamera2()
            self.picam2.configure(
                self.picam2.create_preview_configuration(main={"format": "RGB888", "size": (320, 240)})
            )
            self.picam2.start()
    def close_camera(self):
        self.picam2.stop()
        self.picam2.close()

    def xgoCamera(self,switch):
        global camera_still
        if switch:
            #self.open_camera()
            self.camera_still=True
            t = threading.Thread(target=self.camera_mode)  
            t.start() 
        else:
            self.camera_still=False
            time.sleep(0.5)
            splash = Image.new("RGB",(320,240),"black")
            self.display.ShowImage(splash)
            self.close_camera()

    def camera_mode(self):
        self.camera_still=True
        while 1:
            frame = self.picam2.capture_array() 
            image = cv2.flip(frame, 1)
            b,g,r = cv2.split(image)
            image = cv2.merge((r,g,b))
            image = cv2.flip(image,1)
            imgok = Image.fromarray(image)
            self.display.ShowImage(imgok)
            if not self.camera_still:
                break
    def xgoVideoRecord(self,filename="record",seconds=5):
        path="/home/pi/xgoVideos/"
        self.camera_still=False
        time.sleep(0.6)
        FPS=15
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        width = 320
        height = 240
        videoWrite = cv2.VideoWriter(path+filename+'.mp4', fourcc, FPS, (width,height))
        starttime=time.time()
        while 1:
            print('recording...')
            frame = self.picam2.capture_array() 
            image = cv2.flip(frame, 1)
            # if not ret:
            #     break
            videoWrite.write(image)
            b,g,r = cv2.split(image)
            image = cv2.merge((r,g,b))
            image = cv2.flip(image,1)
            imgok = Image.fromarray(image)
            self.display.ShowImage(imgok)
            if time.time()-starttime>seconds:
                break
        print('recording done')
        self.xgoCamera(True)
        videoWrite.release()

In [2]:
#myrecord：会保存到/home/pi/xgoVideos的文件下  5:录制视频时长
#Myrecord: will be saved to a file in/home/pi/xgoVideos 5: recording video duration
myXGO_edu= Recode_Video()
myXGO_edu.xgoVideoRecord(filename="myvidoes",seconds=5)   


Screen already initialized.


[0:33:43.789655825] [24170]  INFO Camera camera_manager.cpp:325 libcamera v0.3.2+99-1230f78d
[0:33:43.796925709] [24401]  INFO RPI pisp.cpp:695 libpisp version v1.0.7 28196ed6edcf 29-08-2024 (16:33:32)
[0:33:43.806326231] [24401]  INFO RPI pisp.cpp:1154 Registered camera /base/axi/pcie@120000/rp1/i2c@88000/ov5647@36 to CFE device /dev/media2 and ISP device /dev/media0 using PiSP variant BCM2712_D0
[0:33:43.810110155] [24170]  INFO Camera camera.cpp:1197 configuring streams: (0) 320x240-RGB888 (1) 640x480-GBRG_PISP_COMP1
[0:33:43.810212119] [24401]  INFO RPI pisp.cpp:1450 Sensor: /base/axi/pcie@120000/rp1/i2c@88000/ov5647@36 - Selected sensor format: 640x480-SGBRG10_1X10 - Selected CFE format: 640x480-PC1g


recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...
recording...

In [3]:
myXGO_edu.xgoCamera(False)   
del myXGO_edu